#Import

##Import libraries

In [1]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array, array_to_img
from PIL import Image
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import model, encoder, scaler

In [2]:
#Code to load model
loaded_model = load_model('/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000/Exported_Model/my_model.h5')

#Load the pickled encoder
with open('/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000/Exported_Model/one_hot_encoder.pkl', 'rb') as file:
    loaded_encoder = pickle.load(file)

# Load the pickled scaler
with open('/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000/Exported_Model/standard_scaler_pickle.pkl', 'rb') as file:
    loaded_scaler = pickle.load(file)

##Invented data

###Images

In [3]:
folder_path1 = '/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000//Exported_Model/ISIC_0030665.jpg'
folder_path2= '/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000//Exported_Model/ISIC_0026018.jpg'
folder_path3 = '/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000//Exported_Model/ISIC_0027771.jpg'
folder_path4 = '/content/drive/My Drive/Data Science/Master of Data Science and Innovation/MDSI-3S - iLab Capstone Project/Skin_Cancer_MNIST_HAM10000//Exported_Model/ISIC_0033536.jpg'

images = {}
images["image1"] =  Image.open(folder_path1)
images["image2"] =  Image.open(folder_path2)
images["image3"] =  Image.open(folder_path3)
images["image4"] =  Image.open(folder_path4)

image_array = np.array(list(images.values()))

###Dataframe

In [4]:
data = {
    "age": [55, 65, 60, 40],
    "sex": ["female", "male", "female", "male"],
    "localization": ["lower extremity", "back", "back", "abdomen"]
}

df = pd.DataFrame(data)

print(df)

   age     sex     localization
0   55  female  lower extremity
1   65    male             back
2   60  female             back
3   40    male          abdomen


##Encoding

In [5]:
#df is a dataframe that should contain the following columns: ["age", "sex", "localization"]:
#"age": int,
#"sex": 'male', 'female' -- string
#"localization": 'scalp', 'ear', 'face', 'back', 'trunk', 'chest', 'upper extremity', 'abdomen', 'unknown', 'lower extremity', 'genital', 'neck', 'hand', 'foot', 'acral' -- string

encoded_data = loaded_encoder.transform(df[["sex", "localization"]])

# convert array into df with columns names
encoded_df = pd.DataFrame(encoded_data, columns=loaded_encoder.get_feature_names_out(["sex", "localization"]))

# Combine coded columns with numerical columnn
df_coded = pd.concat([df["age"], encoded_df], axis=1)

##Scaling

In [6]:
#Scaling columns
df_normalized = loaded_scaler.transform(df_coded)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [7]:
#Resizing and Normalizing images
def resize_images(images, target_size=(224, 224)):
    resized_images = []
    for image in images:
        resized_image = array_to_img(image).resize(target_size)  # Resize image to 224x224
        image_array = img_to_array(resized_image) / 255.0  # Convert back to array and normalize
        resized_images.append(image_array)
    return np.array(resized_images)

resized_norm_image = resize_images(image_array)

##Predict

In [8]:
def apply_custom_threshold(predictions, threshold=0.6):
    """
    Apply a custom threshold to softmax predictions.
    Predictions are two-class softmax outputs, and this function will return
    the predicted class based on the given threshold.

    Args:
    predictions (np.array): Softmax outputs from the model.
    threshold (float): Threshold for classifying the positive class.

    Returns:
    np.array: Binary classification based on the threshold.
    """
    return (predictions[:, 1] >= threshold).astype(int)  # Assuming class 1 is the positive class

In [9]:
#Code to test model predictions
val_predictions = loaded_model.predict([resized_norm_image, df_normalized])

# Apply the custom threshold
custom_threshold_val_preds = apply_custom_threshold(val_predictions, threshold=0.55)

# resized_norm_image is a numpy array --resized_norm_image.shape = (100,224, 224, 3), where:
#100 = number of pictures
#224 = length
#224 = width
#3 = number of channels in each image

#df_normalized is a numpy array --df_normalized.shape = (100,20), where:
#100 = number of pictures
#20 = number of columns

print("Class 0: not melanoma")
print("Class 1: melanoma")
print(custom_threshold_val_preds)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step
Class 0: not melanoma
Class 1: melanoma
[0 0 0 0]


In [10]:
'class_0', 'class_1'

('class_0', 'class_1')

In [11]:
val_predictions

array([[9.9981147e-01, 1.8849869e-04],
       [5.8905697e-01, 4.1094303e-01],
       [4.7329804e-01, 5.2670199e-01],
       [5.3300107e-01, 4.6699893e-01]], dtype=float32)